## **Data Science Aplicado a las Finanzas** 🚀
### **Examen Final**

Andrés C. Medina Sanhueza

Senior Data Scientist Engineer

anmedinas@gmail.com

Consideraciones Previas

* Examen individual, reemplaza la evaluación final del curso. Debe subir su notebook resuelto a su repositorio personal (se recomienda hacer `commits` a medida que avanza — la trazabilidad del trabajo también se evalúa). Si este, no existe al momento de evaluar, es nota minima, es decir.  1.0.

* Fecha Entrega: martes 22 de Septiembre 2026, 23:59.

* El examen se resuelve **de principio a fin sobre un mismo portafolio**: cada etapa usa resultados de la etapa anterior, así que antes de avanzar revise que el resultado previo tenga sentido (si arrastra un error, se arrastra a todo lo que sigue).

* Fije y declare toda semilla aleatoria (`np.random.seed`, `random_state`, etc.) que utilice, de modo que sus resultados sean reproducibles.

---

### El mandato

Un family office le encarga un informe sobre un portafolio de 4-5 acciones de un mismo sector. El cliente quiere saber tres cosas, en este orden:

1. **¿Cómo se ha comportado cada activo y qué lo explica?** — no basta con mostrar el retorno promedio; el cliente quiere saber si el comportamiento de cada acción responde a factores de mercado identificables (tasas, volatilidad, spread de tasas) o si hay algo idiosincrático.
   
2. **¿El portafolio óptimo que Ud. les recomienda es mejor que uno ingenuo (equiponderado)?** — y si lo es, ¿lo es también cuando la volatilidad de los activos cambia en el tiempo, o solo bajo el supuesto (poco realista) de volatilidad constante?

3. **¿Cuánto podrían perder en un mal mes?** — el cliente necesita una cifra de riesgo (VaR) para el portafolio que Ud. recomiende, y quiere saber si esa cifra cambia si se reconoce que la volatilidad no es constante.

Todo el examen es la construcción de ese informe. Cada etapa que sigue corresponde a una pieza de esa historia, y **el resultado de una etapa es el insumo de la siguiente**: los residuos de la regresión de la Etapa 2 son los que modelará en la Etapa 3; la volatilidad que estime en la Etapa 3 es la que usará como input alternativo en la optimización de la Etapa 4; y el portafolio que optimice en la Etapa 4 es sobre el que calculará el VaR en la Etapa 5. No son ejercicios independientes.

### Portafolio asignado

Cada alumno trabaja sobre un portafolio distinto (mismo ejercicio, distintos datos — no comparen resultados entre ustedes, no deberían coincidir).

| Alumno    | Portafolio (sector)                          | Tickers (Yahoo Finance)              |
|-----------|-----------------------------------------------|----------------------------------------|
| Diego Sanguinettie  | Utilities                                     | `NEE`, `DUK`, `SO`, `AEP`, `EXC`       |
| Edwin Bastias  | Consumo básico (Consumer Staples)             | `PG`, `KO`, `PEP`, `CL`, `MDLZ`        |
| Alvaro Cardenas  | Industriales                                  | `HON`, `UNP`, `CAT`, `GE`, `LMT`       |

Para los tres portafolios, los factores macro a usar en la Etapa 2 son los mismos:

* `GS10` y `GS2` (tasas del Tesoro a 10 y 2 años, FRED)
* `^VIX` (índice de volatilidad, Yahoo Finance)

Use datos **mensuales** entre 2016-01-01 y 2024-01-01 para todo el examen (misma ventana usada en clase, así los resultados son comparables con lo visto en `regression.ipynb`).

---
## Etapa 1 — Radiografía del portafolio

Antes de explicar o predecir nada, necesita conocer los datos. Descargue las 4-5 series de precios de su portafolio asignado y constrúyales un resumen que le permita responder, de un vistazo, "¿cuál de estos activos es el más riesgoso y cuál el más rentable?".

* Descargue los precios de cierre mensuales de su portafolio (2016-01-01 a 2024-01-01) y calcule retornos logarítmicos mensuales para cada activo.
  
* Construya una tabla (un DataFrame, un activo por fila) con: retorno medio anualizado, volatilidad anualizada, coeficiente de asimetría y curtosis.

* Realice el test de Jarque-Bera sobre los retornos de cada activo. ¿Hay evidencia de no-normalidad en alguno de ellos? Esto será relevante más adelante: si los retornos no son normales, cualquier VaR paramétrico que calcule en la Etapa 5 hereda ese problema.

* Grafique la matriz de correlación entre los retornos de los 4-5 activos. Este gráfico es la primera pista de si su "diversificación" dentro del portafolio es real o aparente — guárdelo, lo va a interpretar de nuevo en la Etapa 4.

In [ ]:
# inserte aqui su codigo

---
## Etapa 2 — ¿Qué explica a cada activo?

La Etapa 1 le dio el "qué" (cuánto retorno, cuánto riesgo). Ahora necesita el "por qué": ¿el comportamiento de cada activo responde a factores de mercado observables, o es mayormente idiosincrático?

* Descargue `GS10`, `GS2` (FRED) y `^VIX` (Yahoo) para la misma ventana, y construya el spread `Spread_10Y_2Y = GS10 - GS2`.
  
* Para cada uno de sus 4-5 activos, estime una regresión múltiple: `retorno_activo ~ GS10 + VIX + Spread_10Y_2Y`.

* Reporte el VIF de los regresores. Si detecta multicolinealidad relevante entre `GS10` y `Spread_10Y_2Y` (es esperable, ya que comparten `GS10`), decida y justifique si elimina una variable o las deja ambas.

* Para el activo con mejor $R^2$ y el de peor $R^2$ de su portafolio: revise los supuestos de Gauss-Markov con los gráficos de diagnóstico vistos en clase (residuos vs. ajustados, QQ-plot, Breusch-Pagan). ¿Alguno de los dos viola homocedasticidad o normalidad de forma evidente?

* **Guarde los residuos de cada una de las 4-5 regresiones** — son la parte del retorno de cada activo que los factores macro *no* explican, y es exactamente lo que va a modelar en la Etapa 3. Un activo con residuos grandes y volátiles es un activo cuyo riesgo depende de algo más que del ciclo de tasas o el VIX.

In [ ]:
# inserte aqui su codigo

---
## Etapa 3 — La volatilidad no es constante

En la Etapa 2 modeló la *media* de los retornos con una regresión lineal, que asume implícitamente que la varianza del error es constante en el tiempo (homocedasticidad, supuesto S5 de Gauss-Markov). En finanzas eso rara vez es cierto: la volatilidad se agrupa en el tiempo (períodos tranquilos y períodos turbulentos). Esta etapa cuestiona ese supuesto usando los residuos que guardó en la Etapa 2.

* Para cada uno de los residuos guardados en la Etapa 2, aplique un test ARCH-LM (o inspeccione visualmente el residuo al cuadrado en el tiempo). ¿Hay evidencia de heterocedasticidad condicional?
  
* Ajuste un modelo GARCH(1,1) sobre los residuos de **cada activo** y obtenga la serie de volatilidad condicional estimada $\hat\sigma_t$.

* Compare, para cada activo, la volatilidad condicional del GARCH contra la volatilidad histórica *rolling* de 12 meses (la misma que calculó de forma estática en la Etapa 1, pero ahora en ventana móvil). Grafique ambas series superpuestas.

* Identifique el/los período(s) donde la volatilidad condicional se dispara muy por encima del promedio histórico. ¿Coincide con algún evento de mercado conocido dentro de la ventana 2016-2024?

* **Guarde la volatilidad condicional promedio de cada activo (el último valor de $\hat\sigma_t$, o el promedio de los últimos 12 meses)** — la va a necesitar en la Etapa 4 como una estimación alternativa de riesgo, distinta de la desviación estándar histórica simple.

In [ ]:
# inserte aqui su codigo

---
## Etapa 4 — Dos formas de optimizar el mismo portafolio

Ya tiene dos estimaciones distintas de riesgo por activo: la volatilidad histórica (Etapa 1) y la volatilidad condicional GARCH (Etapa 3). Esta etapa responde la segunda pregunta del cliente: ¿el portafolio óptimo cambia según qué estimación de riesgo se use?

* Construya la frontera eficiente de su portafolio usando la matriz de covarianza histórica (la de siempre, calculada sobre toda la muestra), y obtenga el portafolio de mínima varianza y el de máximo Sharpe.

* Repita el ejercicio, pero ahora reemplace las volatilidades individuales de la diagonal de la matriz de covarianza por las volatilidades condicionales GARCH que guardó en la Etapa 3 (mantenga las correlaciones históricas entre activos — solo cambian las varianzas). Obtenga el nuevo portafolio de máximo Sharpe.

* Compare los pesos de ambos portafolios de máximo Sharpe en una misma tabla o gráfico de barras. ¿Le asigna más o menos peso a los activos que en la Etapa 3 mostraron mayor divergencia entre volatilidad histórica y condicional?

* Vuelva a la matriz de correlación de la Etapa 1: ¿el activo con menor correlación promedio con el resto es también el que recibe más peso en el portafolio de mínima varianza? Comente si el resultado es el esperado según la teoría de diversificación.

In [ ]:
# inserte aqui su codigo

---
## Etapa 5 — ¿Cuánto se puede perder?

Con el portafolio de máximo Sharpe de la Etapa 4 (elija uno de los dos: el histórico o el basado en GARCH, y justifique cuál usa) responda la tercera pregunta del cliente.

* Calcule el VaR al 95% y al 99% del portafolio elegido usando el método **paramétrico** (asumiendo normalidad de los retornos del portafolio) y el método **histórico** (percentil empírico de los retornos simulados del portafolio).

* Calcule también un VaR **condicional**: use la volatilidad GARCH del portafolio (puede aproximarla combinando las $\hat\sigma_t$ de la Etapa 3 con los pesos de la Etapa 4, asumiendo correlaciones históricas constantes) en la fórmula paramétrica del VaR, en vez de la volatilidad histórica constante.

* Presente los tres VaR (paramétrico histórico, histórico empírico, condicional GARCH) en una sola tabla, al 95% y al 99%.

* Retome el resultado de Jarque-Bera de la Etapa 1: si alguno de los activos con mayor peso en el portafolio mostró evidencia de no-normalidad, ¿confía más en el VaR paramétrico o en el histórico? Justifique con lo que vio en esa etapa, no en abstracto.

In [ ]:
# inserte aqui su codigo

---
## Etapa 6 (anexo, 15% de la nota) — ¿Hay régimen de mercado?

A lo largo del examen trató la volatilidad como algo que sube y baja de forma continua (Etapa 3). Una forma alternativa de mirar el mismo problema es preguntarse si el mercado transita entre **regímenes** discretos (por ejemplo, "calma" y "estrés"), y si eso habría cambiado alguna de sus decisiones anteriores.

* Elija **una** de estas dos rutas:
  - **Clustering**: use como *features* mensuales el VIX y la volatilidad condicional promedio de su portafolio (de la Etapa 3), y aplique K-means (o clustering jerárquico) para separar los meses en 2 o 3 regímenes. Grafique la serie de tiempo coloreada por régimen asignado.
  
  - **Supervisado**: construya una variable binaria simple (ej. VIX por sobre o bajo su mediana histórica) y entrene un modelo de clasificación (regresión logística o random forest) que la prediga usando variables rezagadas del propio VIX y del spread de tasas.

* Con el régimen (o la clasificación) obtenido, calcule el VaR histórico de su portafolio (Etapa 5) **por separado** para los meses de "calma" y los meses de "estrés".

* Cierre con un párrafo: si hubiera sabido en qué régimen estaba antes de optimizar (Etapa 4) o de calcular el VaR (Etapa 5), ¿habría tomado una decisión distinta? No hace falta reoptimizar el portafolio — basta con una discusión razonada apoyada en los números que ya obtuvo.

In [ ]:
# inserte aqui su codigo

---
## Informe final

Cierre el notebook con una celda de texto (máximo una página) que responda directamente las tres preguntas del cliente planteadas al inicio, usando los resultados obtenidos en cada etapa como evidencia. No repita cálculos ni gráficos aquí — solo la síntesis.

1. ¿Qué explica el comportamiento de cada activo? (Etapa 2)
   
2. ¿Es mejor el portafolio óptimo que uno equiponderado, y bajo qué supuesto de volatilidad? (Etapas 3-4)

3. ¿Cuánto podría perder el cliente en un mal mes, y qué tan sensible es esa cifra al método usado? (Etapa 5)

---

### Pauta de evaluación

| Criterio               | Descripción                                                                 | Ponderación |
|-------------------------|------------------------------------------------------------------------------|:-----------:|
| Implementación técnica  | Código correcto, supuestos verificados, métodos bien aplicados              | 40% |
| Interpretación          | Lectura económica de cada resultado, conectada con la etapa anterior/siguiente | 30% |
| Reproducibilidad        | El notebook corre de principio a fin, semillas declaradas                   | 15% |
| Informe final           | Responde las 3 preguntas del cliente con evidencia del propio trabajo        | 15% |